Imports

In [ ]:
import json
from pathlib import Path
from typing import Dict, Any, List, Optional
import hashlib
import psycopg2
from psycopg2 import sql

=== Upload inicial das imagens (uma vez) ===
Upload finalizado. Exemplos (primeiros 5): [('C:\\Users\\rodri\\OneDrive - Unesp\\Documentos\\GitHub\\BD2-Trabalho\\CroppedPersonsHighRes\\cam1_trk1_pexels-hikaique-109919_20251113T091521.738215_first.jpg', 55), ('C:\\Users\\rodri\\OneDrive - Unesp\\Documentos\\GitHub\\BD2-Trabalho\\CroppedPersonsHighRes\\cam1_trk2_pexels-hikaique-109919_20251113T091521.738215_first.jpg', 56), ('C:\\Users\\rodri\\OneDrive - Unesp\\Documentos\\GitHub\\BD2-Trabalho\\CroppedPersonsHighRes\\cam1_trk3_pexels-hikaique-109919_20251113T091521.738215_first.jpg', 57), ('C:\\Users\\rodri\\OneDrive - Unesp\\Documentos\\GitHub\\BD2-Trabalho\\CroppedPersonsHighRes\\cam1_trk4_pexels-hikaique-109919_20251113T091521.738215_first.jpg', 58), ('C:\\Users\\rodri\\OneDrive - Unesp\\Documentos\\GitHub\\BD2-Trabalho\\CroppedPersonsHighRes\\cam1_trk5_pexels-hikaique-109919_20251113T091521.738215_first.jpg', 59)]

=== Importar JSONs e linkar imagens existentes (sem re-upload) ===
Res

Configurações globais

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Alterar para suas configurações
cfg_example = {
    "host": "localhost",
    "port": 5432,
    "dbname": "BD2", 
    "user": "postgres",
    "password": "", 
    "images_table": "imagens",
    "items_table": "items",
    "generate_image_url": True,
    "keep_original_source_path": False
}

#Alterar para suas folders
images_folder = "C:\\Users\\rodri\\OneDrive - Unesp\\Documentos\\GitHub\\BD2-Trabalho\\CroppedPersonsHighRes"
jsons_folder = "C:\\Users\\rodri\\OneDrive - Unesp\\Documentos\\GitHub\\BD2-Trabalho\\WithLLM\\OutputJSONHighResQwen8B"

Funções

In [ ]:
# Helpers DB / Hash

# Função para converter nome da imagem em hash
def _sha256_bytes(b: bytes) -> str:
    h = hashlib.sha256()
    h.update(b)
    return h.hexdigest()

# Função para conectar com DB
def connect_db(cfg: Dict[str, Any]):
    return psycopg2.connect(
        host=cfg.get("host", "localhost"),
        port=cfg.get("port", 5432),
        dbname=cfg.get("dbname"),
        user=cfg.get("user"),
        password=cfg.get("password")
    )

# Função para assegurar presença das tabelas e índice para hash
def ensure_tables_and_index(conn, images_table: str = "imagens", items_table: str = "items"):
    cur = conn.cursor()
    cur.execute(
        sql.SQL("""
        CREATE TABLE IF NOT EXISTS {images} (
            id SERIAL PRIMARY KEY,
            filename TEXT,
            content_type TEXT,
            size BIGINT,
            image_hash TEXT NOT NULL,
            data BYTEA,
            created_at TIMESTAMP DEFAULT now()
        );
        """).format(images=sql.Identifier(images_table))
    )
    cur.execute(
        sql.SQL("CREATE UNIQUE INDEX IF NOT EXISTS {idx} ON {images} (image_hash);")
           .format(idx=sql.Identifier(images_table + "_image_hash_idx"),
                   images=sql.Identifier(images_table))
    )
    cur.execute(
        sql.SQL("""
        CREATE TABLE IF NOT EXISTS {items} (
            id SERIAL PRIMARY KEY,
            filename TEXT,
            data jsonb,
            created_at TIMESTAMP DEFAULT now()
        );
        """).format(items=sql.Identifier(items_table))
    )
    conn.commit()
    cur.close()

# Função para realizar upload das imagens a partir de uma folder
def upload_images_folder(images_folder: str, config: Dict[str, Any]) -> Dict[str, int]:
    """
    Faz upload de todas as imagens em images_folder para a tabela de imagens,
    deduplicando por SHA-256. Retorna map: absolute_path -> image_id
    """
    images_dir = Path(images_folder)
    if not images_dir.exists() or not images_dir.is_dir():
        raise FileNotFoundError(f"Imagens folder não existe: {images_folder}")

    cfg = config.copy()
    images_table = cfg.get("images_table", "imagens")
    items_table = cfg.get("items_table", "items")

    conn = connect_db(cfg)
    ensure_tables_and_index(conn, images_table=images_table, items_table=items_table)

    # Abs_path e id
    inserted_map: Dict[str, int] = {}  

    cur = conn.cursor()
    for img in images_dir.rglob("*"):
        if not img.is_file() or img.suffix.lower() not in IMAGE_EXTS:
            continue
        try:
            b = img.read_bytes()
            h = _sha256_bytes(b)
            size = len(b)
            filename = img.name
            suffix = img.suffix.lower()
            content_type = None
            if suffix in {".jpg", ".jpeg"}:
                content_type = "image/jpeg"
            elif suffix == ".png":
                content_type = "image/png"

            insert_q = sql.SQL("""
                INSERT INTO {images} (filename, content_type, size, image_hash, data)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (image_hash) DO NOTHING
                RETURNING id
            """).format(images=sql.Identifier(images_table))

            cur.execute(insert_q, (filename, content_type, size, h, psycopg2.Binary(b)))
            row = cur.fetchone()
            if row:
                image_id = row[0]
                conn.commit()
            else:
                # Já existia, então buscar id
                cur.execute(sql.SQL("SELECT id FROM {images} WHERE image_hash = %s").format(images=sql.Identifier(images_table)), (h,))
                row2 = cur.fetchone()
                if row2:
                    image_id = row2[0]
                else:
                    # Improvável, mas garante consistência
                    raise RuntimeError("Inserção falhou e hash não encontrado")
            inserted_map[str(img.resolve())] = image_id
        except Exception as e:
            # Não interrompe todo o processo, apenas registra
            print(f"[error] upload imagem {img}: {e}")
            conn.rollback()
    cur.close()
    conn.close()
    return inserted_map

# Função para criar e inserir JSONs, vinculando às imagens já existentes no banco
def import_jsons_and_link_images(
    json_folder: str,
    images_folder: str,
    config: Dict[str, Any],
    match_by: List[str] = ["hash", "basename"],
    require_image: bool = True
) -> Dict[str, Any]:

    # Verifica pastas de entrada
    json_dir = Path(json_folder)
    images_dir = Path(images_folder)
    if not json_dir.exists() or not json_dir.is_dir():
        raise FileNotFoundError(f"JSON folder não encontrado: {json_folder}")
    if not images_dir.exists() or not images_dir.is_dir():
        raise FileNotFoundError(f"Images folder não encontrado: {images_folder}")

    # Lê configuração global
    cfg = config.copy()
    images_table = cfg.get("images_table", "imagens")
    items_table = cfg.get("items_table", "items")

    # Conecta ao DB
    conn = connect_db(cfg)
    ensure_tables_and_index(conn, images_table=images_table, items_table=items_table)

    results = {"items_inserted": [], "images_linked": [], "skipped": [], "errors": []}

    # Carrega mapa de imagens do banco (filename e hash)
    cur = conn.cursor()
    cur.execute(sql.SQL(
        "SELECT id, filename, image_hash FROM {images}"
    ).format(images=sql.Identifier(images_table)))
    rows = cur.fetchall()

    filename_to_ids: Dict[str, List[int]] = {}
    hash_to_id: Dict[str, int] = {}

    for rid, fname, ihash in rows:
        if fname:
            filename_to_ids.setdefault(fname, []).append(rid)
        if ihash:
            hash_to_id[ihash] = rid

    cur.close()

    # Função para buscar imagem pelo basename
    def find_by_basename(basename: str) -> Optional[int]:
        ids = filename_to_ids.get(basename)
        return ids[0] if ids else None

    # Processa cada JSON da pasta
    for p in json_dir.iterdir():
        if not p.is_file() or p.suffix.lower() != ".json":
            continue

        # Carrega JSON
        try:
            raw = json.loads(p.read_text(encoding="utf-8"))
        except Exception as e:
            results["errors"].append({"file": str(p), "error": f"JSON parse error: {e}"})
            continue

        # JSON inválido
        if not isinstance(raw, dict):
            results["errors"].append({"file": str(p), "error": "JSON root não é um objeto"})
            continue

        item_filename = p.name
        source_img = raw.get("source_image")
        image_id: Optional[int] = None
        image_url: Optional[str] = None

        # Resolve imagem referenciada
        if isinstance(source_img, str) and any(source_img.lower().endswith(ext) for ext in IMAGE_EXTS):
            cand_path = Path(source_img)

            # Caso 1 — Caminho absoluto válido
            if cand_path.is_absolute() and cand_path.exists() and cand_path.is_file():
                local_candidate = cand_path

            else:
                # Caso 2 — Procurar na pasta de imagens (basename)
                alt1 = images_dir / cand_path.name
                if alt1.exists() and alt1.is_file():
                    local_candidate = alt1
                else:
                    # Caso 3 — Procurar usando caminho relativo
                    alt2 = images_dir / cand_path
                    if alt2.exists() and alt2.is_file():
                        local_candidate = alt2
                    else:
                        # Caso 4 — Procurar na pasta do JSON
                        alt3 = json_dir / cand_path.name
                        if alt3.exists() and alt3.is_file():
                            local_candidate = alt3
                        else:
                            local_candidate = None

            # Estratégia 1 — Match pelo hash
            if local_candidate and "hash" in match_by:
                try:
                    b = local_candidate.read_bytes()
                    h = _sha256_bytes(b)

                    image_id = hash_to_id.get(h)

                    # Se não está no cache local, consulta o banco
                    if image_id is None:
                        cur = conn.cursor()
                        cur.execute(sql.SQL(
                            "SELECT id FROM {images} WHERE image_hash = %s"
                        ).format(images=sql.Identifier(images_table)), (h,))
                        r = cur.fetchone()
                        if r:
                            image_id = r[0]
                            hash_to_id[h] = image_id
                        cur.close()
                except Exception as e:
                    results["errors"].append({
                        "file": str(p),
                        "error": f"erro ao calcular hash: {e}"
                    })

            # Estratégia 2 — Match pelo basename
            if image_id is None and "basename" in match_by:
                image_id = find_by_basename(Path(source_img).name)

        # JSON depende de imagem, mas imagem não encontrada
        if image_id is None and require_image:
            results["skipped"].append({"json_file": str(p), "reason": "image_not_found"})
            print(f"[skip] {p}: imagem não encontrada no DB (source_image={source_img}).")
            continue

        # Monta JSON final
        final_json = dict(raw)
        if image_id is not None:
            final_json["image_id"] = image_id

            if cfg.get("generate_image_url", True):
                image_url = f"db://{images_table}/{image_id}"
                final_json["image_url"] = image_url

            if not cfg.get("keep_original_source_path", False):
                final_json["source_image"] = image_url

        # Valida antes de inserir
        if not isinstance(final_json, dict) or len(final_json) == 0:
            results["errors"].append({"file": str(p), "error": "JSON final vazio"})
            print(f"[error] {p}: final_json vazio — pulando.")
            continue

        # Insere no banco o JSON
        try:
            cur = conn.cursor()
            cur.execute(
                sql.SQL(
                    "INSERT INTO {items} (filename, data) VALUES (%s, %s::jsonb) RETURNING id"
                ).format(items=sql.Identifier(items_table)),
                (item_filename, json.dumps(final_json))
            )
            item_db_id = cur.fetchone()[0]
            conn.commit()
            cur.close()

            results["items_inserted"].append({
                "json_file": str(p),
                "item_id": item_db_id,
                "image_id": image_id
            })
            results["images_linked"].append({
                "json_file": str(p),
                "image_id": image_id
            })

        except Exception as e:
            conn.rollback()
            results["errors"].append({"file": str(p), "error": f"insert error: {e}"})
            print(f"[error] inserindo item {p}: {e}")

    conn.close()
    return results

Inserir Crops

In [ ]:
if __name__ == "__main__":
    print("=== Upload inicial das imagens ===")
    mapping = upload_images_folder(images_folder, cfg_example)
    print("Upload finalizado")

Inserir JSONs referenciando crops inseridos

In [ ]:
if __name__ == "__main__":
    print("\n=== Importar JSONs e linkar imagens existentes (sem re-upload) ===")
    res = import_jsons_and_link_images(jsons_folder, images_folder, cfg_example, require_image=True)
    print("Resumo da importação:", res)